In [2]:


# ===========================
# Imports & global configuration
# ===========================

from __future__ import annotations

import uuid
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from faker import Faker


# We explicitly fix a random seed to make the dataset generation
# reproducible. This is important for debugging, experimentation,
# and ensuring that later analysis notebooks remain consistent.
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# We use the Brazilian locale because SynaptiqPay is a Brazilian fintech.
# This gives us realistic-looking names, emails, and locations.
fake = Faker("pt_BR")
Faker.seed(RANDOM_SEED)


# ===========================
# High-level dataset design
# ===========================

# Total number of customers described in the summary.
N_CUSTOMERS = 10_000

# Planted customer segments (ground truth) with their target sizes.
SEGMENT_DISTRIBUTION = {
    "high_value_active": 2_000,   # 20%
    "mid_value_regular": 3_000,   # 30%
    "low_value_dormant": 3_000,   # 30%
    "at_risk_churner": 2_000,     # 20%
}

# Sanity check: guarantee the distribution adds up to the total count.
assert sum(SEGMENT_DISTRIBUTION.values()) == N_CUSTOMERS

# Possible acquisition channels from the project summary.
ACQUISITION_CHANNELS = ["paid_ads", "organic", "referral", "partnership"]

# We consider registrations over a recent 24-month window so that
# cohort analysis and tenure-related features make sense.
TODAY = datetime(2026, 3, 1)
REGISTRATION_START = TODAY - timedelta(days=730)  # ~24 months ago

# Gross margin used in the LTV calculation.
GROSS_MARGIN = 0.60


# ===========================
# Helper functions for feature generation
# ===========================

def random_registration_date() -> datetime:
    """Sample a registration date uniformly within the last 24 months."""

    # We draw a random offset (in days) from the start of the window.
    days_offset = np.random.randint(0, (TODAY - REGISTRATION_START).days + 1)
    return REGISTRATION_START + timedelta(days=int(days_offset))


def choice_with_probs(options, probs):
    """Convenience wrapper for numpy choice with explicit probabilities."""

    return np.random.choice(options, p=probs)


def generate_customer_row(true_segment: str) -> dict:
    """Generate a single synthetic customer row for a given true segment.

    The logic in this function encodes the behavioral profile of each
    planted segment. All downstream analysis (RFM, clustering, churn
    modeling, etc.) should be able to recover these patterns.
    """

    # -------------------------
    # Identity & metadata
    # -------------------------
    customer_id = str(uuid.uuid4())
    name = fake.name()
    email = fake.email()
    age = int(np.random.normal(loc=35, scale=10))
    age = int(np.clip(age, 18, 80))  # basic realism constraint

    # We use Brazilian states for location realism.
    state = fake.estado_sigla()

    registration_date = random_registration_date()

    # Acquisition channels may have slightly different probabilities by segment
    # to seed interesting patterns for unit economics analysis.
    if true_segment == "high_value_active":
        acq_channel = choice_with_probs(
            ACQUISITION_CHANNELS,
            probs=[0.30, 0.20, 0.30, 0.20],  # more likely paid_ads / referral
        )
        acquisition_cost = float(np.random.normal(180, 40))
    elif true_segment == "mid_value_regular":
        acq_channel = choice_with_probs(
            ACQUISITION_CHANNELS,
            probs=[0.25, 0.30, 0.25, 0.20],
        )
        acquisition_cost = float(np.random.normal(150, 35))
    elif true_segment == "low_value_dormant":
        acq_channel = choice_with_probs(
            ACQUISITION_CHANNELS,
            probs=[0.20, 0.40, 0.20, 0.20],  # more organic / lower CAC
        )
        acquisition_cost = float(np.random.normal(110, 25))
    else:  # at_risk_churner
        acq_channel = choice_with_probs(
            ACQUISITION_CHANNELS,
            probs=[0.35, 0.15, 0.30, 0.20],  # aggressive paid + referral
        )
        acquisition_cost = float(np.random.normal(190, 45))

    acquisition_cost = float(max(acquisition_cost, 20.0))  # lower bound

    # -------------------------
    # Financial behavior
    # -------------------------
    if true_segment == "high_value_active":
        avg_monthly_balance = np.random.normal(10_000, 3_000)
        monthly_transactions_count = np.random.poisson(35)
        avg_ticket = np.random.normal(220, 60)
        credit_limit = np.random.normal(25_000, 5_000)
        credit_utilization_pct = np.random.normal(45, 15)
    elif true_segment == "mid_value_regular":
        avg_monthly_balance = np.random.normal(5_000, 1_500)
        monthly_transactions_count = np.random.poisson(18)
        avg_ticket = np.random.normal(160, 40)
        credit_limit = np.random.normal(15_000, 3_000)
        credit_utilization_pct = np.random.normal(40, 15)
    elif true_segment == "low_value_dormant":
        avg_monthly_balance = np.random.normal(800, 400)
        monthly_transactions_count = np.random.poisson(4)
        avg_ticket = np.random.normal(90, 30)
        credit_limit = np.random.normal(6_000, 2_000)
        credit_utilization_pct = np.random.normal(20, 10)
    else:  # at_risk_churner
        avg_monthly_balance = np.random.normal(200, 150)
        monthly_transactions_count = np.random.poisson(1)
        avg_ticket = np.random.normal(70, 25)
        credit_limit = np.random.normal(8_000, 3_000)
        credit_utilization_pct = np.random.normal(10, 8)

    # Ensure non-negative values and realistic ranges.
    avg_monthly_balance = float(max(avg_monthly_balance, 0))
    monthly_transactions_count = int(max(monthly_transactions_count, 0))
    avg_ticket = float(max(avg_ticket, 5))
    credit_limit = float(max(credit_limit, 500))
    credit_utilization_pct = float(np.clip(credit_utilization_pct, 0, 120))

    # Aggregate 12-month flows around the monthly behavior.
    total_spent_12m = float(max(np.random.normal(
        monthly_transactions_count * avg_ticket * 12,
        monthly_transactions_count * avg_ticket * 4,
    ), 0))
    total_deposits_12m = float(max(np.random.normal(
        avg_monthly_balance * 12 * 0.8,
        avg_monthly_balance * 12 * 0.3,
    ), 0))

    # Revenue NovaPay generates from this customer, tied loosely to balance
    # and transactional activity.
    base_revenue = (
        0.004 * avg_monthly_balance  # spread / float revenue
        + 0.01 * monthly_transactions_count * avg_ticket  # interchange + fees
    )
    avg_monthly_revenue = float(max(np.random.normal(base_revenue, base_revenue * 0.25), 5.0))

    # -------------------------
    # Recency & engagement
    # -------------------------
    if true_segment == "high_value_active":
        days_since_last_transaction = np.random.randint(0, 15)
        days_since_last_login = np.random.randint(0, 7)
        monthly_app_logins = np.random.poisson(28)
        support_tickets_6m = np.random.poisson(1)
    elif true_segment == "mid_value_regular":
        days_since_last_transaction = np.random.randint(5, 30)
        days_since_last_login = np.random.randint(3, 20)
        monthly_app_logins = np.random.poisson(16)
        support_tickets_6m = np.random.poisson(1)
    elif true_segment == "low_value_dormant":
        days_since_last_transaction = np.random.randint(30, 120)
        days_since_last_login = np.random.randint(20, 120)
        monthly_app_logins = np.random.poisson(4)
        support_tickets_6m = np.random.poisson(0.5)
    else:  # at_risk_churner
        days_since_last_transaction = np.random.randint(90, 365)
        days_since_last_login = np.random.randint(60, 365)
        monthly_app_logins = np.random.poisson(2)
        support_tickets_6m = np.random.poisson(0.3)

    days_since_last_transaction = int(max(days_since_last_transaction, 0))
    days_since_last_login = int(max(days_since_last_login, 0))
    monthly_app_logins = int(max(monthly_app_logins, 0))
    support_tickets_6m = int(max(support_tickets_6m, 0))

    # -------------------------
    # Product ownership
    # -------------------------
    if true_segment == "high_value_active":
        products_owned = np.random.randint(3, 6)
        has_credit_card = True
        has_investments = np.random.rand() < 0.65
        has_insurance = np.random.rand() < 0.45
        has_loans = np.random.rand() < 0.35
    elif true_segment == "mid_value_regular":
        products_owned = np.random.randint(2, 5)
        has_credit_card = np.random.rand() < 0.9
        has_investments = np.random.rand() < 0.35
        has_insurance = np.random.rand() < 0.30
        has_loans = np.random.rand() < 0.30
    elif true_segment == "low_value_dormant":
        products_owned = np.random.randint(1, 3)
        has_credit_card = np.random.rand() < 0.6
        has_investments = np.random.rand() < 0.15
        has_insurance = np.random.rand() < 0.20
        has_loans = np.random.rand() < 0.20
    else:  # at_risk_churner
        products_owned = np.random.randint(0, 3)
        has_credit_card = np.random.rand() < 0.5
        has_investments = np.random.rand() < 0.1
        has_insurance = np.random.rand() < 0.15
        has_loans = np.random.rand() < 0.25

    products_owned = int(max(products_owned, 0))

    # -------------------------
    # Ground-truth churn probability (used to plant signal)
    # -------------------------
    # We seed churn probability based on segment, then allow the
    # downstream logistic regression model to learn these patterns.
    if true_segment == "high_value_active":
        churn_probability = np.random.beta(1.2, 12)  # very low
    elif true_segment == "mid_value_regular":
        churn_probability = np.random.beta(1.6, 8)
    elif true_segment == "low_value_dormant":
        churn_probability = np.random.beta(3.0, 5)
    else:  # at_risk_churner
        churn_probability = np.random.beta(6.0, 2)  # very high

    churn_probability = float(np.clip(churn_probability, 0.01, 0.99))

    # LTV-related derived values will be computed after we assemble
    # the full DataFrame, but we already have the ingredients here.

    return {
        # Identity & metadata
        "customer_id": customer_id,
        "name": name,
        "email": email,
        "age": age,
        "state": state,
        "registration_date": registration_date,
        "acquisition_channel": acq_channel,
        "acquisition_cost": acquisition_cost,
        # Financial behavior
        "avg_monthly_balance": avg_monthly_balance,
        "avg_transaction_ticket": avg_ticket,
        "monthly_transactions_count": monthly_transactions_count,
        "total_spent_12m": total_spent_12m,
        "total_deposits_12m": total_deposits_12m,
        "credit_limit": credit_limit,
        "credit_utilization_pct": credit_utilization_pct,
        "avg_monthly_revenue": avg_monthly_revenue,
        # Recency & engagement
        "days_since_last_transaction": days_since_last_transaction,
        "days_since_last_login": days_since_last_login,
        "monthly_app_logins": monthly_app_logins,
        "support_tickets_6m": support_tickets_6m,
        # Product ownership
        "products_owned": products_owned,
        "has_credit_card": has_credit_card,
        "has_investments": has_investments,
        "has_insurance": has_insurance,
        "has_loans": has_loans,
        # Ground truth segment + churn (used later for validation / training)
        "true_segment": true_segment,
        "churn_probability": churn_probability,
    }


# ===========================
# Generate base rows for all segments
# ===========================

rows = []

for segment_label, segment_size in SEGMENT_DISTRIBUTION.items():
    for _ in range(segment_size):
        rows.append(generate_customer_row(segment_label))

# Build the initial DataFrame with all core columns plus true_segment
# and churn_probability.
df = pd.DataFrame(rows)


# ===========================
# Derived columns (schema-aligned)
# ===========================

# Tenure in months: how long the customer has been with SynaptiqPay.
# We compute this from registration_date and "today".

def _compute_tenure_months(registration_date: pd.Timestamp) -> int:
    delta_days = (TODAY - registration_date.to_pydatetime()).days
    # Approximate months as 30 days; detailed accuracy is unnecessary here.
    return max(int(delta_days // 30), 0)


df["registration_date"] = pd.to_datetime(df["registration_date"])
df["tenure_months"] = df["registration_date"].apply(_compute_tenure_months)

# Cohort month: used for cohort analysis and time-based cuts.
df["cohort_month"] = df["registration_date"].dt.to_period("M").astype(str)

# Recency score: map days_since_last_transaction to a 1–5 score
# (1 = very old, 5 = very recent).
recency_bins = [-1, 7, 30, 90, 180, np.inf]
recency_labels = [5, 4, 3, 2, 1]
df["recency_score"] = pd.cut(
    df["days_since_last_transaction"],
    bins=recency_bins,
    labels=recency_labels,
).astype(int)

# Frequency score: map monthly_transactions_count to a 1–5 scale.
frequency_bins = [-1, 2, 8, 20, 35, np.inf]
frequency_labels = [1, 2, 3, 4, 5]
df["frequency_score"] = pd.cut(
    df["monthly_transactions_count"],
    bins=frequency_bins,
    labels=frequency_labels,
).astype(int)

# Monetary score: map avg_monthly_revenue to a 1–5 scale.
monetary_bins = [-1, 50, 150, 350, 700, np.inf]
monetary_labels = [1, 2, 3, 4, 5]
df["monetary_score"] = pd.cut(
    df["avg_monthly_revenue"],
    bins=monetary_bins,
    labels=monetary_labels,
).astype(int)

# Aggregate RFM score: we use the mean of the three components to obtain
# a smooth score between 1 and 5. Downstream analysis can discretize it
# again as needed.
df["rfm_score"] = df[["recency_score", "frequency_score", "monetary_score"]].mean(axis=1)


# Cohort retention rate placeholder
# ---------------------------------
# For the synthetic dataset we approximate cohort retention rates using
# a simple function that decreases with tenure but allows variability
# by segment type. This provides a realistic-looking feature for
# visualization and for the churn model to use.

def _approximate_cohort_retention(row) -> float:
    tenure = row["tenure_months"]
    segment = row["true_segment"]

    # Base decay curve: rapid early churn, then slower.
    base_rate = np.exp(-0.08 * tenure)

    # Segment adjustments: high_value cohorts retain better, at_risk worse.
    if segment == "high_value_active":
        base_rate *= 1.25
    elif segment == "mid_value_regular":
        base_rate *= 1.05
    elif segment == "low_value_dormant":
        base_rate *= 0.85
    else:  # at_risk_churner
        base_rate *= 0.70

    # Clip to a realistic [0, 1] interval.
    return float(np.clip(base_rate, 0.05, 0.99))


df["cohort_retention_rate"] = df.apply(_approximate_cohort_retention, axis=1)


# LTV and unit economics features
# -------------------------------
# ltv = avg_monthly_revenue × (1 / churn_probability) × gross_margin
# ltv_cac_ratio = ltv / acquisition_cost
# payback_period_months = acquisition_cost / avg_monthly_revenue

ltv = df["avg_monthly_revenue"] * (1.0 / df["churn_probability"]) * GROSS_MARGIN

# For numerical stability and realism, we avoid extremely high values
# by clipping the long tail.
df["ltv"] = ltv.clip(upper=100_000)

df["ltv_cac_ratio"] = (df["ltv"] / df["acquisition_cost"]).replace([np.inf, -np.inf], np.nan)
df["ltv_cac_ratio"] = df["ltv_cac_ratio"].clip(upper=50).fillna(0.0)

payback = df["acquisition_cost"] / df["avg_monthly_revenue"]
df["payback_period_months"] = payback.clip(upper=120)


# Predicted segment placeholder
# -----------------------------
# This column is reserved for the K-Means clustering output that will be
# computed in a later notebook. We initialize it as NaN so that the
# schema is already complete.
df["predicted_segment"] = pd.Series([np.nan] * len(df))


# ===========================
# Final sanity checks & preview
# ===========================

expected_columns = [
    # Identity & metadata
    "customer_id",
    "name",
    "email",
    "age",
    "state",
    "registration_date",
    "acquisition_channel",
    "acquisition_cost",
    # Financial behavior
    "avg_monthly_balance",
    "avg_transaction_ticket",
    "monthly_transactions_count",
    "total_spent_12m",
    "total_deposits_12m",
    "credit_limit",
    "credit_utilization_pct",
    "avg_monthly_revenue",
    # Recency & engagement
    "days_since_last_transaction",
    "days_since_last_login",
    "monthly_app_logins",
    "support_tickets_6m",
    # Product ownership
    "products_owned",
    "has_credit_card",
    "has_investments",
    "has_insurance",
    "has_loans",
    # Derived scores and labels
    "tenure_months",
    "recency_score",
    "frequency_score",
    "monetary_score",
    "rfm_score",
    "cohort_month",
    "cohort_retention_rate",
    "predicted_segment",
    "true_segment",
    "ltv",
    "ltv_cac_ratio",
    "payback_period_months",
    "churn_probability",
]

missing_cols = set(expected_columns) - set(df.columns)
extra_cols = set(df.columns) - set(expected_columns)

print(f"Total customers: {len(df):,}")
print(f"Missing columns relative to schema: {missing_cols}")
print(f"Extra columns not listed in schema: {extra_cols}")

# Display a small sample to visually inspect that the dataset
# looks consistent with the business narrative and schema.
df.head()


Total customers: 10,000
Missing columns relative to schema: set()
Extra columns not listed in schema: set()


,customer_id,name,email,age,state,registration_date,acquisition_channel,acquisition_cost,avg_monthly_balance,avg_transaction_ticket,...,cohort_month,recency_score,frequency_score,monetary_score,rfm_score,cohort_retention_rate,ltv,ltv_cac_ratio,payback_period_months,predicted_segment
0,0eef1173-e3e6-431a-8c8d-acd6a52a7540,Brenda Alves,samuel32@example.net,39,AM,2024-06-15,referral,174.469428,6406.580720,348.499506,...,2024-06,5,4,2,3.666667,0.252371,888.423362,5.092143,1.418339,NaN
1,59fe9208-ebef-4265-a560-312f8bd1e62d,Ísis Borges,bandrade@example.org,33,DF,2025-09-18,partnership,225.373571,9685.763355,98.396823,...,2025-09,5,5,2,4.000000,0.837900,736.859469,3.269503,2.707659,NaN
2,f7457110-b946-487a-ba44-dac712e15c60,Thiago Pereira,novaismiguel@example.net,24,RS,2025-02-01,referral,116.668231,11382.448900,225.155408,...,2025-02,4,5,2,3.666667,0.441818,2471.513311,21.184116,1.486372,NaN
3,1688146f-6bef-4c9e-8384-cb6738f7908f,Dra. Mariah Câmara,rpastor@example.org,47,SP,2026-02-24,partnership,138.486805,9772.588765,309.726055,...,2026-02,4,5,3,4.000000,0.990000,500.215289,3.612007,0.879556,NaN
4,8e8c2138-758c-49d8-bc4a-b14671dfe934,Otávio Fernandes,leonardo35@example.org,28,AP,2024-03-28,paid_ads,167.353364,12887.923870,207.418454,...,2024-03,4,4,2,3.333333,0.198522,1407.814410,8.412227,1.813779,NaN
